In [1]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# 🕵️ Choosing a Replacement Strategy

Four replacement strategies compared side-by-side on the same data.

#### 📚 What you'll learn

- Compare **Redact**, **Annotate**, **Hash**, and **Substitute** on the same input
- Customize output formats with `format_template`
- Understand which strategy fits your use case (readability, determinism, privacy)

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import all four strategy classes: `Redact`, `Annotate`, `Hash`, `Substitute`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Annotate, Anonymizer, AnonymizerConfig, AnonymizerInput, Hash, Redact, Substitute

In [4]:
anonymizer = Anonymizer()

[18:15:47] [INFO] 🔧 Anonymizer initialized with 3 model configs


[18:15:47] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[18:15:47] [INFO]   |-- ✅ validator: gpt-oss-120b


[18:15:47] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- We use the same biographies dataset throughout so each strategy is compared
  on identical input.

In [5]:
input_data = AnonymizerInput(
    source="../data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## 🚫 Redact

- Replaces each entity with a label-based marker. Default: `[REDACTED_FIRST_NAME]`.
- Customize with `Redact(format_template=...)`.

In [6]:
redact_config = AnonymizerConfig(replace=Redact())

redact_preview = anonymizer.preview(
    config=redact_config,
    data=input_data,
    num_records=3,
)

redact_preview.display_record(0)

[18:15:47] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:15:47] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:15:47] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:15:47] [INFO] 🔍 Running entity detection on 3 records


[18:16:40] [INFO]   |-- 📋 Detection complete — 74 entities found across 3 records (0 failed) [52.5s]


[18:16:40] [INFO]   |-- labels: first_name=23, state=6, company_name=6, age=5, occupation=5, city=5, education_level=4, last_name=3, race_ethnicity=3, language=3, political_view=2, religious_belief=2, street_address=2, school_name=1, university_name=1, date_of_birth=1, product_name=1, employment_status=1


[18:16:40] [INFO] 🔄 Running Redact replacement


[18:16:40] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:16:40] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[REDACTED_FIRST_NAME]
Watford,last_name,[REDACTED_LAST_NAME]
40,age,[REDACTED_AGE]
Mexican,race_ethnicity,[REDACTED_RACE_ETHNICITY]
veterinarian,occupation,[REDACTED_OCCUPATION]
Denver,city,[REDACTED_CITY]
Colorado,state,[REDACTED_STATE]
Jefferson High,school_name,[REDACTED_SCHOOL_NAME]
DVM,education_level,[REDACTED_EDUCATION_LEVEL]
University of Colorado Boulder,university_name,[REDACTED_UNIVERSITY_NAME]


### Custom template

- `format_template="***"` replaces every entity with the same constant.

In [7]:
custom_config = AnonymizerConfig(replace=Redact(format_template="***"))

custom_preview = anonymizer.preview(
    config=custom_config,
    data=input_data,
    num_records=3,
)

custom_preview.display_record(0)

[18:16:40] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:16:40] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:16:40] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:16:40] [INFO] 🔍 Running entity detection on 3 records


[18:17:22] [INFO]   |-- 📋 Detection complete — 74 entities found across 3 records (0 failed) [42.4s]


[18:17:22] [INFO]   |-- labels: first_name=23, state=6, company_name=6, age=5, occupation=5, city=5, education_level=4, last_name=3, race_ethnicity=3, language=3, political_view=3, street_address=2, school_name=1, date_of_birth=1, division_name=1, project_name=1, employment_status=1, religious_belief=1


[18:17:22] [INFO] 🔄 Running Redact replacement


[18:17:22] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:17:22] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,***
Watford,last_name,***
40,age,***
Mexican,race_ethnicity,***
veterinarian,occupation,***
Denver,city,***
Colorado,state,***
Jefferson High,school_name,***
DVM,education_level,***
University of Colorado Boulder,company_name,***


## 🏷️ Annotate

- Tags each entity with its label but keeps the original text visible.
  Default: `<Alice, first_name>`.
- Customize with `format_template` -- must include `{text}` and `{label}`,
  e.g. `Annotate(format_template="<{text}-|-{label}>")`.

In [8]:
annotate_config = AnonymizerConfig(replace=Annotate())

annotate_preview = anonymizer.preview(
    config=annotate_config,
    data=input_data,
    num_records=3,
)

annotate_preview.display_record(0)

[18:17:22] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:17:22] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:17:22] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:17:22] [INFO] 🔍 Running entity detection on 3 records


[18:18:27] [INFO]   |-- 📋 Detection complete — 71 entities found across 3 records (0 failed) [65.3s]


[18:18:27] [INFO]   |-- labels: first_name=22, state=6, company_name=6, age=5, occupation=5, city=5, education_level=4, last_name=3, race_ethnicity=3, language=3, political_view=2, street_address=2, school_name=1, university_name=1, date_of_birth=1, employment_status=1, religious_belief=1


[18:18:27] [INFO] 🔄 Running Annotate replacement


[18:18:27] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:18:27] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,"<Bobby, first_name>"
Watford,last_name,"<Watford, last_name>"
40,age,"<40, age>"
Mexican,race_ethnicity,"<Mexican, race_ethnicity>"
veterinarian,occupation,"<veterinarian, occupation>"
Denver,city,"<Denver, city>"
Colorado,state,"<Colorado, state>"
Jefferson High,school_name,"<Jefferson High, school_name>"
DVM,education_level,"<DVM, education_level>"
University of Colorado Boulder,university_name,"<University of Colorado Boulder, university_name>"


### Custom template

- Override the default format with any string containing `{text}` and `{label}`.

In [9]:
annotate_custom_config = AnonymizerConfig(replace=Annotate(format_template="<{text}-|-{label}>"))
annotate_custom_preview = anonymizer.preview(
    config=annotate_custom_config,
    data=input_data,
    num_records=3,
)
annotate_custom_preview.display_record(0)

[18:18:27] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:18:27] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:18:27] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:18:27] [INFO] 🔍 Running entity detection on 3 records


[18:19:10] [INFO]   |-- 📋 Detection complete — 73 entities found across 3 records (0 failed) [43.2s]


[18:19:10] [INFO]   |-- labels: first_name=22, company_name=7, state=6, age=5, occupation=5, city=5, education_level=4, last_name=3, race_ethnicity=3, language=3, political_view=3, street_address=2, school_name=1, university_name=1, date_of_birth=1, employment_status=1, religious_belief=1


[18:19:10] [INFO] 🔄 Running Annotate replacement


[18:19:10] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:19:10] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<Bobby-|-first_name>
Watford,last_name,<Watford-|-last_name>
40,age,<40-|-age>
Mexican,race_ethnicity,<Mexican-|-race_ethnicity>
veterinarian,occupation,<veterinarian-|-occupation>
Denver,city,<Denver-|-city>
Colorado,state,<Colorado-|-state>
Jefferson High,school_name,<Jefferson High-|-school_name>
DVM,education_level,<DVM-|-education_level>
University of Colorado Boulder,university_name,<University of Colorado Boulder-|-university_name>


## #️⃣ Hash

- Deterministic -- same input always produces the same hash.
- Customize `format_template` (must include `{digest}`),
  `algorithm` (`sha256`/`sha1`/`md5`), and `digest_length` (6--64).

In [10]:
hash_config = AnonymizerConfig(replace=Hash())

hash_preview = anonymizer.preview(
    config=hash_config,
    data=input_data,
    num_records=3,
)

hash_preview.display_record(0)

[18:19:10] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:19:10] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:19:10] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:19:10] [INFO] 🔍 Running entity detection on 3 records


[18:20:12] [INFO]   |-- 📋 Detection complete — 71 entities found across 3 records (0 failed) [61.5s]


[18:20:12] [INFO]   |-- labels: first_name=22, city=6, state=6, age=5, occupation=5, education_level=4, last_name=3, race_ethnicity=3, language=3, company_name=3, political_view=3, street_address=2, school_name=1, university_name=1, clinic_name=1, date_of_birth=1, employment_status=1, religious_belief=1


[18:20:12] [INFO] 🔄 Running Hash replacement


[18:20:12] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:20:12] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<HASH_FIRST_NAME_4a70dab2cb4d>
Watford,last_name,<HASH_LAST_NAME_e2efa8a62600>
40,age,<HASH_AGE_d59eced1ded0>
Mexican,race_ethnicity,<HASH_RACE_ETHNICITY_d108dfd1df5c>
veterinarian,occupation,<HASH_OCCUPATION_52a469e4d8e9>
Denver,city,<HASH_CITY_fcdeb8c07d4a>
Colorado,state,<HASH_STATE_4ae62bf4e804>
Jefferson High,school_name,<HASH_SCHOOL_NAME_39dde416149c>
DVM,education_level,<HASH_EDUCATION_LEVEL_d44ae5e206d1>
University of Colorado,university_name,<HASH_UNIVERSITY_NAME_533be36b77e3>


### Custom template

- Override the algorithm, digest length, and output format.

In [11]:
hash_custom_config = AnonymizerConfig(replace=Hash(algorithm="md5", digest_length=8, format_template="[{digest}]"))
hash_custom_preview = anonymizer.preview(
    config=hash_custom_config,
    data=input_data,
    num_records=3,
)
hash_custom_preview.display_record(0)

[18:20:12] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:20:12] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:20:12] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:20:12] [INFO] 🔍 Running entity detection on 3 records


[18:21:13] [INFO]   |-- 📋 Detection complete — 75 entities found across 3 records (0 failed) [60.8s]


[18:21:13] [INFO]   |-- labels: first_name=22, state=7, city=6, company_name=6, age=5, occupation=5, education_level=4, political_view=4, last_name=3, race_ethnicity=3, language=3, street_address=2, date_of_birth=1, facility_name=1, project_name=1, employment_status=1, religious_belief=1


[18:21:13] [INFO] 🔄 Running Hash replacement


[18:21:13] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[18:21:13] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[657b3da9]
Watford,last_name,[6e424e2c]
40,age,[d645920e]
Mexican,race_ethnicity,[a0e769d8]
veterinarian,occupation,[84c99b4a]
Denver,city,[67100af8]
Colorado,state,[15e49475]
DVM,education_level,[47211f54]
Boulder,city,[3c62c506]
English,language,[78463a38]


## 🔄 Substitute

- Uses an LLM to generate contextually appropriate synthetic replacements.
- Unlike the strategies above, the LLM considers the full document context --
  matching names with emails, cities to states, etc.
- Customize with `instructions` to steer the LLM's replacement choices.

In [12]:
substitute_config = AnonymizerConfig(replace=Substitute())

substitute_preview = anonymizer.preview(
    config=substitute_config,
    data=input_data,
    num_records=3,
)

[18:21:13] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:21:13] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:21:13] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:21:13] [INFO] 🔍 Running entity detection on 3 records


[18:21:59] [INFO]   |-- 📋 Detection complete — 70 entities found across 3 records (0 failed) [46.0s]


[18:21:59] [INFO]   |-- labels: first_name=22, state=6, age=5, occupation=5, city=5, education_level=4, company_name=4, last_name=3, race_ethnicity=3, language=3, political_view=2, religious_belief=2, street_address=2, school_name=1, university_name=1, date_of_birth=1, employment_status=1


[18:21:59] [INFO] 🔄 Running Substitute replacement


[18:22:16] [INFO]   |-- 📋 Replacement complete (0 failed) [17.3s]


[18:22:16] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


In [13]:
substitute_preview.display_record(0)

Original,Label,Replacement
40,age,45
Aria and Leo,first_name,Nina and Omar
Bobby,first_name,Ethan
Christian Democrat,political_view,Libertarian
Colorado,state,Oregon
Colorado Veterinary Clinic,company_name,Cascade Animal Care
DVM,education_level,Bachelor of Science
Denver,city,Portland
English,language,Spanish
Jefferson High,school_name,Lincoln Secondary


### Custom instructions

- Pass `instructions` to guide the LLM -- e.g. keep replacements within
  a specific region, culture, or naming convention.

In [14]:
substitute_custom_config = AnonymizerConfig(
    replace=Substitute(instructions="Use only Japanese names and locations for all replacements.")
)
substitute_custom_preview = anonymizer.preview(
    config=substitute_custom_config,
    data=input_data,
    num_records=3,
)
substitute_custom_preview.display_record(0)

[18:22:16] [INFO] 📂 Loaded 25 records from ../data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[18:22:16] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[18:22:16] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:22:16] [INFO] 🔍 Running entity detection on 3 records


[18:23:08] [INFO]   |-- 📋 Detection complete — 72 entities found across 3 records (0 failed) [52.0s]


[18:23:08] [INFO]   |-- labels: first_name=22, state=6, company_name=6, age=5, occupation=5, city=5, education_level=4, last_name=3, race_ethnicity=3, language=3, political_view=2, street_address=2, high_school_name=1, university_name=1, date_of_birth=1, project_name=1, employment_status=1, religious_belief=1


[18:23:08] [INFO] 🔄 Running Substitute replacement


[18:23:27] [INFO]   |-- 📋 Replacement complete (0 failed) [18.7s]


[18:23:27] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
40,age,52
Aria and Leo,first_name,Sakura and Haru
Bobby,first_name,Takumi
Christian Democrat,political_view,Liberal Democratic Party
Colorado,state,Kyoto
Colorado Veterinary Clinic,company_name,Kyoto Animal Hospital
DVM,education_level,MSc
Denver,city,Uji
English,language,Japanese
Jefferson High,high_school_name,Kawasaki High School


## ⏭️ Next steps

- **[🕵️ Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  dig into what the detection pipeline found and debug quality.
- **[✏️ Rewriting Biographies](04_rewriting_biographies.ipynb)** --
  generate privacy-safe paraphrases instead of token-level replacements.
- **[⚖️ Rewriting Legal Documents](05_rewriting_legal_documents.ipynb)** --
  rewrite legal text with domain-specific privacy goals.